# Двухканальный детектор Deepfake-изображений (RGB + FFT)
Основная идея заключается в одновременном анализе пространственных признаков и артефактов в частотной области. Входное изображение проходит через два параллельных пути обработки, что позволяет улавливать как визуальные несоответствия, так и специфические следы работы генеративных алгоритмов.

Первая ветка сети обрабатывает стандартные RGB-каналы, извлекая высокоуровневые семантические признаки. Вторая ветка работает с амплитудным спектром, полученным через быстрое преобразование Фурье (FFT) и логарифмическую нормализацию. Для повышения эффективности в обе ветки интегрированы блоки внимания CBAM, которые помогают модели фокусироваться на наиболее значимых деталях и каналах. Основу архитектуры составляют остаточные блоки (ResBlocks), обеспечивающие стабильное обучение глубокой сети.

Ниже представлена структурная схема модели:
```
Входное изображение [B, 3, 224, 224]
                 │
        ┌────────┴────────┐
        │                 │
   Ветка RGB        Ветка FFT (Спектр)
        │                 │
   Conv + CBAM       FFT + Log Norm
        │                 │
   ResBlock x N      ResBlock x M
        │                 │
   Global Pool       Global Pool
        │                 │
     feat_rgb          feat_fft
        │                 │
        └────────┬────────┘
                 │
          Concat [768 + 768]
                 │
        Linear (1024) + SiLU
                 │
         Linear (1) + Sigmoid
```

Система поддерживает гибкое переключение между стандартной кросс-энтропией и Focal Loss для лучшей обработки сложных примеров. Для оптимизации используется AdamW с планировщиком скорости обучения, который реагирует на плато валидационной метрики. Финальные предсказания строятся с применением техники Test-Time Augmentation (TTA), объединяющей результаты для нескольких вариаций одного изображения. Модель автоматически подбирает оптимальный порог классификации для максимизации метрики F1 на валидационной выборке. Весь процесс сопровождается детальной логгизацией и сохранением результатов в формате Parquet для дальнейшего анализа.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import math
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
import os
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from google.colab import drive
import zipfile
import gc

In [ ]:
# --- 1. СПЕКТРАЛЬНЫЕ И ЧАСТОТНЫЕ ФИЛЬТРЫ ---
def extract_fft(x):
    x_gray = x.mean(dim=1, keepdim=True)

    fft = torch.fft.fft2(x_gray)
    fft_shifted = torch.fft.fftshift(fft)

    magnitude = torch.abs(fft_shifted)
    log_mag = torch.log1p(magnitude)

    # нормализация
    log_mag = (log_mag - log_mag.mean()) / (log_mag.std() + 1e-6)

    return log_mag.repeat(1, 3, 1, 1)

def extract_laplacian(x):
    """Выделяет текстуры и шум фильтром Лапласа"""
    # Ядро Лапласа
    kernel = torch.tensor([[0., 1., 0.],
                           [1., -4., 1.],
                           [0., 1., 0.]], device=x.device).view(1, 1, 3, 3)
    kernel = kernel.repeat(x.size(1), 1, 1, 1) # Для каждого канала
    return F.conv2d(x, kernel, padding=1, groups=x.size(1))

# --- 2. FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduce=True):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduce = reduce

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss) # Вероятность правильного класса
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss

        if self.reduce:
            return torch.mean(F_loss)
        else:
            return F_loss


In [ ]:

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1   = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv1(x_cat))

class CBAMBlock(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAMBlock, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

# --- 4. RESIDUAL BLOCK С CBAM ---
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_cbam=True):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.use_cbam = use_cbam
        if use_cbam:
            self.cbam = CBAMBlock(out_ch)

        self.shortcut = nn.Sequential()
        # Избегаем TypeError со skip_connections при несовпадении размерностей
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_cbam:
            out = self.cbam(out)
        out += self.shortcut(x)
        return F.relu(out)

# --- 5. MINIBATCH STDDEV (Из GAN) ---
class MinibatchStdDev(nn.Module):
    def forward(self, x):
        b, c, h, w = x.shape
        std = torch.std(x, dim=0, keepdim=True) # (1, C, H, W)
        mean_std = torch.mean(std, dim=(1,2,3), keepdim=True)
        mean_std = mean_std.expand(b, 1, h, w)
        return torch.cat([x, mean_std], dim=1)


In [ ]:
class DeepFakeDetector(nn.Module):
    def __init__(self):
        super(DeepFakeDetector, self).__init__()

        # --- ВЕТКА 1: RGB (Глубокая, 512 признаков) ---
        self.rgb_branch = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(3, 2, 1),
            ResBlock(64, 128, stride=2),
            ResBlock(128, 256, stride=2),
            ResBlock(256, 512, stride=2),
            # MinibatchStdDev(), # Вход 512, выход 513
            nn.Conv2d(512, 512, 3, padding=1),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

        # --- ВЕТКА 2: FFT (Менее глубокая, сохраняем пики, 256) ---
        self.fft_branch = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            ResBlock(64, 128, stride=2, use_cbam=False),
            ResBlock(128, 256, stride=2, use_cbam=False),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

        # --- ОБУЧАЕМЫЕ ВЕСА FUSION ---
        self.alpha = nn.Parameter(torch.tensor(1.0))
        self.beta = nn.Parameter(torch.tensor(1.0))

        # --- ФИНАЛЬНЫЙ КЛАССИФИКАТОР ---
        self.classifier = nn.Sequential(
            nn.Linear(512 + 256, 128),  # RGB + FFT
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x_fft = extract_fft(x)

        f_rgb = self.rgb_branch(x)
        f_fft = self.fft_branch(x_fft)

        # веса
        f_rgb = f_rgb * self.alpha
        f_fft = f_fft * self.beta

        f = torch.cat([f_rgb, f_fft], dim=1)

        logits = self.classifier(f)
        return logits.squeeze(-1)


In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.df.iloc[idx]['Id']) + '.jpg'
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = float(self.df.iloc[idx]['target_feature'])
        return image, label

# Аугментации (БЕЗ геометрии, только артефакты)
train_transforms = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

val_transforms = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])


def get_sampler(df_train):
    class_counts = df_train['target_feature'].value_counts().to_dict()
    num_samples = len(df_train)
    # Вес = 1 / количество примеров в классе
    class_weights = {cls: num_samples / count for cls, count in class_counts.items()}
    sample_weights = [class_weights[label] for label in df_train['target_feature']]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler


In [ ]:
def find_best_threshold(y_true, y_probs):
    best_thresh = 0.5
    best_f1 = 0.0
    thresholds = np.linspace(0.0, 1.0, 100)

    for thresh in thresholds:
        y_pred = (y_probs >= thresh).astype(int)
        f1 = f1_score(y_true, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    return best_thresh, best_f1

import os

def train_pipeline(model, train_loader, val_loader, epochs=15, save_path='/content/drive/MyDrive/best_model_weights_30_epoch.pth'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)


    best_val_f1 = -1.0
    best_threshold_overall = 0.0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            noise = torch.randn_like(images) * 0.02
            images = images + noise

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Валидация
        model.eval()
        val_probs = []
        val_targets = []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                logits = model(images)
                probs = torch.sigmoid(logits)

                val_probs.extend(probs.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())

        val_probs = np.array(val_probs)
        val_targets = np.array(val_targets)

        # Динамический поиск порога
        best_thresh, current_val_f1 = find_best_threshold(val_targets, val_probs)
        val_preds = (val_probs >= best_thresh).astype(int)

        recall = recall_score(val_targets, val_preds)
        precision = precision_score(val_targets, val_preds)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss/len(train_loader):.4f}")
        print(f"-> Val F1: {current_val_f1:.4f} | Recall: {recall:.4f} | Prec: {precision:.4f} | Best Thresh: {best_thresh:.2f}")

        scheduler.step(current_val_f1) # Шагаем по шедулеру в зависимости от F1

        # Сохраняем модель, если текущий F1 лучше предыдущего
        if current_val_f1 > best_val_f1:
            best_val_f1 = current_val_f1
            best_threshold_overall = best_thresh
            torch.save(model.state_dict(), save_path)
            print(f"⭐ Новая лучшая модель сохранена с F1: {best_val_f1:.4f} в {save_path}")


    return model, best_threshold_overall


---------------------------
-------------
____________________________
----------

----------------------
_______________________________


In [ ]:
import torch
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive
import os
import zipfile
import gc

print("🚀 Шаг 1: Монтирование Google Drive и распаковка архива...")
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/ml-intensive-yandex-academy-spring-2026.zip'
extract_path = '/content/dataset_root'

!unzip -q "{zip_path}" -d "{extract_path}"

# Определяем пути к данным
BASE_PATH = os.path.join(extract_path, 'dataset')
TRAIN_IMG_PATH = os.path.join(BASE_PATH, 'train_images')
TEST_IMG_PATH = os.path.join(BASE_PATH, 'test_images')
SOLUTION_PATH = os.path.join(BASE_PATH, 'train_solution.csv')
print("✅ Данные распакованы.")

🚀 Шаг 1: Монтирование Google Drive и распаковка архива...
Mounted at /content/drive
unzip:  cannot find or open /content/drive/MyDrive/ml-intensive-yandex-academy-spring-2026.zip, /content/drive/MyDrive/ml-intensive-yandex-academy-spring-2026.zip.zip or /content/drive/MyDrive/ml-intensive-yandex-academy-spring-2026.zip.ZIP.
✅ Данные распакованы.


In [ ]:
# --- ШАГ 2: ЗАГРУЗКА РЕШЕНИЙ И РАЗДЕЛЕНИЕ НА TRAIN/VAL ---
print("\n🚀 Шаг 2: Загрузка CSV и разделение данных на train/validation...")
full_train_df = pd.read_csv(SOLUTION_PATH, header=None)
full_train_df.columns = ['Id', 'target_feature']

print("Первые 5 строк train_solution.csv:")
print(full_train_df.head())
print("\nНазвания колонок в train_solution.csv:")
print(full_train_df.columns)

# Разделяем данные 80/20. `stratify` гарантирует, что в train и val
# будет одинаковое процентное соотношение real/fake, как и в исходном датасете.
train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.2,
    random_state=42,
    stratify=full_train_df['target_feature']
)
print(f"Размер Train: {len(train_df)} | Размер Validation: {len(val_df)}")
print("✅ Данные разделены.")


🚀 Шаг 2: Загрузка CSV и разделение данных на train/validation...
Первые 5 строк train_solution.csv:
   Id  target_feature
0   0               0
1   1               1
2   2               1
3   3               0
4   4               0

Названия колонок в train_solution.csv:
Index(['Id', 'target_feature'], dtype='object')
Размер Train: 40000 | Размер Validation: 10000
✅ Данные разделены.


In [ ]:

# --- ШАГ 3: СОЗДАНИЕ DATASET'ОВ И DATALOADER'ОВ ---
print("\n🚀 Шаг 3: Создание Dataset'ов и DataLoader'ов...")

train_dataset = DeepfakeDataset(df=train_df, img_dir=TRAIN_IMG_PATH, transform=train_transforms)
val_dataset = DeepfakeDataset(df=val_df, img_dir=TRAIN_IMG_PATH, transform=val_transforms)

train_sampler = get_sampler(train_df)

BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)
print("✅ DataLoader'ы готовы.")


🚀 Шаг 3: Создание Dataset'ов и DataLoader'ов...
✅ DataLoader'ы готовы.


In [ ]:
# --- ШАГ 4: ЗАПУСК ПРОЦЕССА ОБУЧЕНИЯ ---
print("\n🚀 Шаг 4: Запуск процесса обучения...")
torch.cuda.empty_cache()
gc.collect()

# Инициализируем модель
model = DeepFakeDetector()

trained_model, best_threshold = train_pipeline(
    model,
    train_loader,
    val_loader,
    epochs=30 # Можете изменить количество эпох
)
print(f"✅ Обучение завершено! Лучший F1-порог для валидации: {best_threshold:.4f}")


🚀 Шаг 4: Запуск процесса обучения...
Epoch [1/10] | Train Loss: 0.4093
-> Val F1: 0.5765 | Recall: 0.6271 | Prec: 0.5335 | Best Thresh: 0.51
⭐ Новая лучшая модель сохранена с F1: 0.5765 в /content/drive/MyDrive/best_model_weights.pth
Epoch [2/10] | Train Loss: 0.2846
-> Val F1: 0.6946 | Recall: 0.6882 | Prec: 0.7010 | Best Thresh: 0.13
⭐ Новая лучшая модель сохранена с F1: 0.6946 в /content/drive/MyDrive/best_model_weights.pth
Epoch [3/10] | Train Loss: 0.1974
-> Val F1: 0.7973 | Recall: 0.7694 | Prec: 0.8273 | Best Thresh: 0.42
⭐ Новая лучшая модель сохранена с F1: 0.7973 в /content/drive/MyDrive/best_model_weights.pth
Epoch [4/10] | Train Loss: 0.1303
-> Val F1: 0.8123 | Recall: 0.8029 | Prec: 0.8218 | Best Thresh: 0.11
⭐ Новая лучшая модель сохранена с F1: 0.8123 в /content/drive/MyDrive/best_model_weights.pth
Epoch [5/10] | Train Loss: 0.0889
-> Val F1: 0.8184 | Recall: 0.8176 | Prec: 0.8191 | Best Thresh: 0.06
⭐ Новая лучшая модель сохранена с F1: 0.8184 в /content/drive/MyDrive/

In [ ]:
drive.mount('/content/drive')

In [ ]:
trained_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_weights.pth'))
print("✅ Загружена лучшая модель с валидационным F1=0.8599")

✅ Загружена лучшая модель с валидационным F1=0.8599


In [ ]:
# --- ШАГ 5: ИНФЕРЕНС НА ТЕСТОВЫХ ДАННЫХ И СОЗДАНИЕ SUBMISSION ---
print("\n🚀 Шаг 5: Формирование файла для сабмишена...")

test_files = os.listdir(TEST_IMG_PATH)
test_ids = [int(f.split('.')[0]) for f in test_files]
test_df = pd.DataFrame({'Id': test_ids, 'target_feature': 0})

test_dataset = DeepfakeDataset(df=test_df, img_dir=TEST_IMG_PATH, transform=val_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Переводим модель в режим инференса
trained_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_predictions = []

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        logits = trained_model(images)
        probs = torch.sigmoid(logits)
        preds = (probs >= best_threshold).int()
        test_predictions.extend(preds.cpu().numpy())


submission_df = pd.DataFrame({'Id': test_ids, 'target_feature': test_predictions})
submission_df = submission_df.sort_values('Id').reset_index(drop=True)

submission_df.to_csv('submission30.csv', index=False)

print("✅ Файл 'submission.csv' успешно создан и готов к отправке!")



🚀 Шаг 5: Формирование файла для сабмишена...
✅ Файл 'submission.csv' успешно создан и готов к отправке!


# tta предсказания


In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
trained_model = DeepFakeDetector()
trained_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_weights.pth'))
print("✅ Загружена лучшая модель с валидационным F1=0.8599")

✅ Загружена лучшая модель с валидационным F1=0.8599


In [ ]:
best_threshold = 0.35

In [ ]:
BATCH_SIZE = 32

In [ ]:
def predict_with_tta(model, images, device):
    """
    Возвращает вероятности класса 1, усреднённые по трём вариантам:
    - исходное изображение
    - горизонтальное отражение
    - поворот на 180°
    """
    model.eval()
    with torch.no_grad():
        # 1) Оригинал
        logits_orig = model(images.to(device))
        probs_orig = torch.sigmoid(logits_orig)

        # 2) Горизонтальное отражение (flip left-right)
        flipped = torch.flip(images, dims=[3])
        logits_flip = model(flipped.to(device))
        probs_flip = torch.sigmoid(logits_flip)

        # 3) Поворот на 180 градусов
        rotated = torch.rot90(images, k=2, dims=(2, 3))  # дважды повернуть на 90°
        logits_rot = model(rotated.to(device))
        probs_rot = torch.sigmoid(logits_rot)

        probs_avg = (probs_orig + probs_flip + probs_rot) / 3.0
    return probs_avg

In [ ]:
# --- ШАГ 5: ИНФЕРЕНС НА ТЕСТОВЫХ ДАННЫХ И СОЗДАНИЕ SUBMISSION ---
print("\n🚀 Шаг 5: Формирование файла для сабмишена...")

test_files = os.listdir(TEST_IMG_PATH)
test_ids = [int(f.split('.')[0]) for f in test_files]
test_df = pd.DataFrame({'Id': test_ids, 'target_feature': 0})

test_dataset = DeepfakeDataset(df=test_df, img_dir=TEST_IMG_PATH, transform=val_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

trained_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

trained_model = trained_model.to(device)

test_predictions = []

with torch.no_grad():
    for images, _ in test_loader:
        probs = predict_with_tta(trained_model, images, device)
        preds = (probs >= best_threshold).int()
        test_predictions.extend(preds.cpu().numpy())

submission_df = pd.DataFrame({'id': test_ids, 'target_feature': test_predictions})
submission_df = submission_df.sort_values('id').reset_index(drop=True)

submission_df.to_csv('submission.csv', index=False)

print("✅ Файл 'submission.csv' успешно создан и готов к отправке!")



🚀 Шаг 5: Формирование файла для сабмишена...
✅ Файл 'submission.csv' успешно создан и готов к отправке!


# создание .parquet файлов

In [ ]:
zip_path = '/content/drive/MyDrive/FFT/ml-intensive-yandex-academy-spring-2026.zip'
extract_path = '/content/dataset_root'

!unzip -q "{zip_path}" -d "{extract_path}"

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# --- 1. КОНФИГУРАЦИЯ ---
MODEL_NAME = 'RGB_FFT'
WEIGHTS_PATH = '/content/drive/MyDrive/ALL_MODELS/weights/RGB_FFT/rgb_fft.pth'
DRIVE_SAVE_PATH = f'/content/drive/MyDrive/ALL_MODELS/{MODEL_NAME}'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Path(DRIVE_SAVE_PATH).mkdir(parents=True, exist_ok=True)

# --- 2. ДАТАСЕТ ---
class SimpleInferenceDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row['Id'])
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        img = Image.open(img_path).convert('RGB')
        target = float(row.get('target_feature', -1))
        return self.transform(img), img_id, torch.tensor(target, dtype=torch.float32)

# Трансформы
val_transforms = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# --- 3. ИНИЦИАЛИЗАЦИЯ ---
print(f"Загрузка модели {MODEL_NAME}...")
model = DeepFakeDetector().to(DEVICE)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

# --- 4. ИНФЕРЕНС ---
def run_inference(df, img_dir, is_test=False):
    ds = SimpleInferenceDataset(df, img_dir, val_transforms)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)

    all_ids, all_logits, all_probs, all_targets = [], [], [], []

    with torch.no_grad():
        for imgs, ids, targets in tqdm(loader, desc=f"Inference {MODEL_NAME}"):
            imgs = imgs.to(DEVICE)

            # БЕЗ TTA: прямой проход
            outputs = model(imgs)
            logits = outputs.reshape(-1)
            probs = torch.sigmoid(logits)

            all_ids.extend(ids)
            all_logits.extend(logits.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            if not is_test: all_targets.extend(targets.numpy())

    res = pd.DataFrame({'id': all_ids, 'logit': all_logits, 'prob': all_probs})
    if not is_test: res['true_label'] = all_targets

    def maybe_int(s): return int(s) if str(s).isdigit() else s
    res['id_sort'] = res['id'].apply(maybe_int)
    res = res.sort_values('id_sort').drop(columns=['id_sort']).reset_index(drop=True)
    return res

# --- 5. ЗАПУСК ---
full_df = pd.read_csv('/content/dataset_root/dataset/train_solution.csv', header=None, names=['Id', 'target_feature'])
full_df['Id'] = full_df['Id'].astype(str)
_, val_df = train_test_split(full_df, test_size=0.2, random_state=42, stratify=full_df['target_feature'])

test_path = Path('/content/dataset_root/dataset/test_images')
test_ids = [p.stem for p in test_path.glob('*.jpg')]
test_df = pd.DataFrame({'Id': test_ids})

print("\n--- Валидация ---")
val_res = run_inference(val_df, '/content/dataset_root/dataset/train_images')
val_res.to_parquet(f"{MODEL_NAME}_val_preds.parquet")
!cp {MODEL_NAME}_val_preds.parquet {DRIVE_SAVE_PATH}/

print("\n--- Тест ---")
test_res = run_inference(test_df, '/content/dataset_root/dataset/test_images', is_test=True)
test_res.to_parquet(f"{MODEL_NAME}_test_preds.parquet")
!cp {MODEL_NAME}_test_preds.parquet {DRIVE_SAVE_PATH}/

print(f"\nГотово! Результаты сохранены в {DRIVE_SAVE_PATH}")

Загрузка модели RGB_FFT...

--- Валидация ---


Inference RGB_FFT: 100%|██████████| 157/157 [00:35<00:00,  4.44it/s]



--- Тест ---


Inference RGB_FFT: 100%|██████████| 157/157 [00:32<00:00,  4.84it/s]



Готово! Результаты сохранены в /content/drive/MyDrive/ALL_MODELS/RGB_FFT
